# Notebook #10: Niche Discovery

This notebook uses CellCharter to identify shared spatial immune niches across endometriosis lesions.

For v1, there are 2 peritoneal lesions.

**Main questions:**
1. What spatial niches exist within each lesion?
2. Are the same niches present in both lesions?
3. What cell types define each niche?

In [176]:
from google.colab import drive
drive.mount('/content/drive', force_remount = True)

Mounted at /content/drive


In [177]:
!pip install scanpy\
squidpy\
numpy==2.0.2\
pandas\
matplotlib\
seaborn\
cellcharter\
lightning\
scikit-learn\
anndata

In [178]:
# -- Imports
from pathlib import Path

import scanpy as sc
import squidpy as sq

import numpy as np
import pandas as pd
import seaborn as sns
import cellcharter as cc
import matplotlib.pyplot as plt
from lightning.pytorch import seed_everything
from sklearn.preprocessing import StandardScaler
from matplotlib.patches import Patch

import anndata as ad
ad.settings.allow_write_nullable_strings = True

In [179]:
# -- Paths
project_dir = Path(
    "/content/drive/MyDrive/endo-immune-atlas"
)

input_path_346 = (
    project_dir
    / "data"
    / "interim"
    / "spatial"
    / "BEME_346_spatial_immunosenescence.h5ad"
)

input_path_355G = (
    project_dir
    / "data"
    / "interim"
    / "spatial"
    / "BEME_355G_spatial_immunosenescence.h5ad"
)

output_path_data = (
    project_dir
    / "data"
    / "interim"
    / "spatial"
)

output_path_figures = (
    project_dir
    / "figures"
    / "spatial"
    / "niche_discovery"
)

output_path_results = (
    project_dir
    / "results"
    / "spatial"
    / "niche_discovery"
)

output_path_data.mkdir(
    parents=True,
    exist_ok=True
)

output_path_figures.mkdir(
    parents=True,
    exist_ok=True
)

output_path_results.mkdir(
    parents=True,
    exist_ok=True
)

In [180]:
# -- SET THE SEED
seed_everything(3)

INFO: Seed set to 3
INFO:lightning.fabric.utilities.seed:Seed set to 3


3

In [181]:
# -- Palette
niche_palette = {
    "Immune-depleted": "#D9D9D9",
    "Effector immune": "#A9643A",
    "Lymphoid-rich": "#D8B04C",
    "Immune hotspot": "#8E1F2F",
    "Immune-cold": "#A8BED1",
    "Adaptive-enriched": "#B45E82",
    "Diffuse immune": "#6F9272",
}

In [182]:
# -- Import objects
adata_346 = sc.read_h5ad(
    input_path_346
)

adata_355G = sc.read_h5ad(
    input_path_355G
)

adatas = {
    "BEME_346": adata_346,
    "BEME_355G": adata_355G
}

for sample_id, adata in adatas.items():
    print(
        sample_id,
        adata.shape
    )

BEME_346 (1388, 22220)
BEME_355G (1960, 22220)


Since the goal is to compare the neighborhoods between the tissue samples, the tissue objects will be grouped together to make sure that Niche 1 in 346 is the same niche in 355G. This also will make life easier if more tissues are added to the analysis pipeline.

In [183]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# FUNCTION_10_1 -- PREP AND CLEAN OBJECT BEFORE ANALYSIS
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def prepare_cellcharter_adata(
    adata,
    sample_id,
    abundance_key="q05_cell_abundance_w_sf"
):
    """Create a lightweight copy containing CellCharter inputs."""

    if abundance_key not in adata.obsm:
        raise KeyError(
            f"{abundance_key} is missing from {sample_id}."
        )

    adata_cc = adata.copy()

    abundance = (
        adata_cc.obsm[abundance_key]
        .copy()
    )

    # -- Remove cell2location prefix from labels
    abundance.columns = [
        column.split("_sf_", 1)[-1]
        for column in abundance.columns
    ]

    cell_types = abundance.columns.tolist()

    adata_cc.obsm[
        "immune_abundance"
    ] = abundance.to_numpy()

    adata_cc.uns[
        "immune_cell_types"
    ] = cell_types

    adata_cc.obs[
        "sample_id"
    ] = sample_id

    adata_cc.obs[
        "tissue_type"
    ] = "endometriosis"

    adata_cc.obs[
        "lesion_site"
    ] = "peritoneal"

    keep_obsm = {
        "spatial",
        "immune_abundance",
        abundance_key
    }

    for key in list(
        adata_cc.obsm.keys()
    ):
        if key not in keep_obsm:
            del adata_cc.obsm[key]

    return adata_cc


adatas_cc = {
    sample_id: prepare_cellcharter_adata(
        adata,
        sample_id=sample_id
    )
    for sample_id, adata
    in adatas.items()
}

cell_type_sets = {
    sample_id: tuple(
        adata.uns[
            "immune_cell_types"
        ]
    )
    for sample_id, adata
    in adatas_cc.items()
}

if len(
    set(
        cell_type_sets.values()
    )
) != 1:
    raise ValueError(
        "Spatial objects do not contain the same "
        "cell2location cell-type columns."
    )

cell_types = list(
    next(
        iter(
            cell_type_sets.values()
        )
    )
)

print(
    "Cell types:",
    cell_types
)

Cell types: ['B', 'CD4 T', 'CD8 T', 'DC', 'Mono-C', 'Mono-NC', 'NK-CD16+', 'NK-CD16-', 'TRM', 'Treg', 'γδ T']


In [184]:
# -- Concat lesions

processed = []

spatial_uns = {}

for sample_id, adata in adatas_cc.items():

    sample = adata.copy()

    sample.obs_names_make_unique()

    sample.obs_names = [
        f"{sample_id}_{spot}"
        for spot in sample.obs_names
    ]

    sample.obs[
        "sample_id"
    ] = sample_id

    # -- Preserve sample-specific spatial metadata
    if "spatial" in sample.uns:
        spatial_uns.update(
            sample.uns["spatial"]
        )

    processed.append(
        sample
    )

adata_immune = ad.concat(
    processed,
    join="outer",
    merge="same",
    label=None,
    index_unique=None
)

adata_immune.uns[
    "spatial"
] = spatial_uns

adata_immune.obs[
    "sample_id"
] = adata_immune.obs[
    "sample_id"
].astype("category")

adata_immune.uns[
    "immune_cell_types"
] = cell_types

print(adata_immune)
print(
    adata_immune.obs[
        "sample_id"
    ].value_counts()
)

AnnData object with n_obs × n_vars = 3348 × 22220
    obs: 'in_tissue', 'array_row', 'array_col', 'library_id', 'sample_id', 'gsm_id', 'tissue_type', 'condition', 'lesion_site', '_indices', '_scvi_batch', '_scvi_labels', 'meanscell_abundance_w_sf_B', 'meanscell_abundance_w_sf_CD4 T', 'meanscell_abundance_w_sf_CD8 T', 'meanscell_abundance_w_sf_DC', 'meanscell_abundance_w_sf_Mono-C', 'meanscell_abundance_w_sf_Mono-NC', 'meanscell_abundance_w_sf_NK-CD16+', 'meanscell_abundance_w_sf_NK-CD16-', 'meanscell_abundance_w_sf_TRM', 'meanscell_abundance_w_sf_Treg', 'meanscell_abundance_w_sf_γδ T', 'Total immune', 'B', 'CD4 T', 'CD8 T', 'DC', 'Mono-C', 'Mono-NC', 'NK-CD16+', 'NK-CD16-', 'TRM', 'Treg', 'γδ T', 'leiden_0.2', 'leiden_0.4', 'leiden_0.6', 'leiden_0.8', 'region_cluster', 'total_immune_spot_level', 'total_immune_region_level', 'tissue_level_immune_class', 'projected_senescence_score', 'cluster_mean_senescence', 'projected_dysfunction_score', 'cluster_mean_dysfunction'
    var: 'gene_ids',

In [185]:
# -- Scale abundance & build neighborhoods
adata_immune.obsm[
    "immune_abundance_scaled"
] = StandardScaler().fit_transform(
    adata_immune.obsm[
        "immune_abundance"
    ]
)

sq.gr.spatial_neighbors(
    adata_immune,
    coord_type="grid",
    n_neighs=6,
    library_key="sample_id"
)

cc.gr.remove_long_links(
    adata_immune
)

cc.gr.aggregate_neighbors(
    adata_immune,
    use_rep="immune_abundance_scaled",
    n_layers=3
)

for key in adata_immune.obsm.keys():
    print(
        key,
        adata_immune.obsm[key].shape
    )

INFO     Creating graph using `None` transform and `2` libraries.                                                  


/tmp/ipykernel_3418/3658328562.py:10: FutureWarning: Calling `spatial_neighbors` is deprecated and will be removed in squidpy v1.9.0. Use `spatial_neighbors_knn`, `spatial_neighbors_radius`, `spatial_neighbors_delaunay`, `spatial_neighbors_grid`, or `spatial_neighbors_from_builder` instead.
  sq.gr.spatial_neighbors(


  0%|          | 0/4 [00:00<?, ?it/s]

q05_cell_abundance_w_sf (3348, 11)
spatial (3348, 2)
immune_abundance (3348, 11)
immune_abundance_scaled (3348, 11)
X_cellcharter (3348, 44)


In [186]:
# -- CellCharter Niche Discovery
model = cc.tl.Cluster(
    n_clusters=7,
    random_state=3
)

model.fit(
    adata_immune,
    use_rep="X_cellcharter"
)

adata_immune.obs["immune_niche"] = model.predict(
    adata_immune,
    use_rep="X_cellcharter"
)

adata_immune.obs[
    "immune_niche"
] = adata_immune.obs[
    "immune_niche"
].astype("category")

print(
    adata_immune.obs[
        "immune_niche"
    ].value_counts()
)

display(
    pd.crosstab(
        adata_immune.obs[
            "immune_niche"
        ],
        adata_immune.obs[
            "sample_id"
        ]
    )
)

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: False, used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Output()

immune_niche
1    759
4    577
5    570
0    548
6    507
2    300
3     87
Name: count, dtype: int64


sample_id,BEME_346,BEME_355G
immune_niche,,
0,66,482
1,759,0
2,0,300
3,0,87
4,563,14
5,0,570
6,0,507


In [187]:
# -- Composition tables
immune_df = pd.DataFrame(
    adata_immune.obsm[
        "immune_abundance"
    ],
    columns=cell_types,
    index=adata_immune.obs_names
)

immune_df[
    "immune_niche"
] = (
    adata_immune.obs[
        "immune_niche"
    ]
    .astype(str)
    .values
)

immune_df[
    "sample_id"
] = (
    adata_immune.obs[
        "sample_id"
    ]
    .astype(str)
    .values
)

niche_composition = (
    immune_df
    .groupby(
        "immune_niche"
    )[cell_types]
    .mean()
)

global_mean = (
    immune_df[
        cell_types
    ]
    .mean()
)

celltype_enrichment = (
    niche_composition
    .div(
        global_mean
    )
)

niche_counts = (
    adata_immune.obs[
        "immune_niche"
    ]
    .value_counts()
    .sort_index()
)

immune_state_niche = pd.crosstab(
    adata_immune.obs[
        "immune_niche"
    ],
    adata_immune.obs[
        "tissue_level_immune_class"
    ],
    normalize="index"
)

niche_sample = pd.crosstab(
    adata_immune.obs[
        "immune_niche"
    ],
    adata_immune.obs[
        "sample_id"
    ],
    normalize="columns"
)

display(niche_composition)
display(celltype_enrichment)
display(immune_state_niche)
display(niche_sample)

,B,CD4 T,CD8 T,DC,Mono-C,Mono-NC,NK-CD16+,NK-CD16-,TRM,Treg,γδ T
immune_niche,,,,,,,,,,,
0,0.157146,0.387686,0.516947,0.044708,0.038949,0.027349,0.143381,0.235416,0.241051,0.258731,0.076317
1,2.473986,7.219265,20.941978,2.389093,1.067098,1.925037,3.096264,6.506348,5.180392,15.435795,1.535707
2,10.069570,24.105631,25.028278,3.173606,1.553994,2.278169,6.958303,14.664486,9.732594,14.428214,5.737482
3,10.010875,7.133041,42.580608,6.430794,2.971523,4.382323,15.169409,31.069443,16.288073,29.258299,12.778070
4,1.316900,4.064203,6.064246,0.740905,0.424380,0.686623,0.773053,1.556850,1.850752,3.976080,0.350709
5,4.027764,13.745680,6.677948,0.686420,0.309040,0.401890,1.081591,2.394937,2.837794,2.787275,0.855303
6,4.649219,7.881999,15.861521,2.173966,0.949032,1.487649,5.125470,10.532853,6.887209,10.259260,4.206532


,B,CD4 T,CD8 T,DC,Mono-C,Mono-NC,NK-CD16+,NK-CD16-,TRM,Treg,γδ T
immune_niche,,,,,,,,,,,
0,0.046690,0.046824,0.040496,0.028401,0.053049,0.023357,0.050546,0.039861,0.055360,0.031142,0.037230
1,0.735049,0.871925,1.640529,1.517680,1.453374,1.644011,1.091530,1.101651,1.189728,1.857897,0.749174
2,2.991781,2.911419,1.960637,2.016045,2.116520,1.945592,2.453019,2.482983,2.235186,1.736621,2.798953
3,2.974342,0.861511,3.335632,4.085185,4.047179,3.742572,5.347689,5.260661,3.740716,3.521613,6.233610
4,0.391266,0.490864,0.475054,0.470662,0.578000,0.586387,0.272525,0.263605,0.425043,0.478572,0.171088
5,1.196693,1.660169,0.523130,0.436051,0.420908,0.343221,0.381295,0.405509,0.651727,0.335484,0.417248
6,1.381335,0.951968,1.242542,1.381020,1.292570,1.270475,1.806888,1.783417,1.581715,1.234834,2.052100


tissue_level_immune_class,Immune-high,Immune-low,Immune-moderate,Immune-moderate-high,Immune-moderate-low,Immune-poor
immune_niche,,,,,,
0,0.000000,0.001825,0.000000,0.000000,0.000000,0.998175
1,0.376812,0.169960,0.000000,0.188406,0.251647,0.013175
2,0.130000,0.083333,0.086667,0.473333,0.226667,0.000000
3,0.885057,0.000000,0.022989,0.080460,0.011494,0.000000
4,0.017331,0.410745,0.000000,0.024263,0.159445,0.388215
5,0.000000,0.449123,0.001754,0.117544,0.266667,0.164912
6,0.151874,0.258383,0.284024,0.049310,0.248521,0.007890


sample_id,BEME_346,BEME_355G
immune_niche,,
0,0.04755,0.245918
1,0.54683,0.000000
2,0.00000,0.153061
3,0.00000,0.044388
4,0.40562,0.007143
5,0.00000,0.290816
6,0.00000,0.258673


In [188]:
# -- Niche abundance
fig, ax = plt.subplots(
    figsize=(7, 4)
)

sns.barplot(
    x=niche_counts.index.astype(str),
    y=niche_counts.values,
    ax=ax
)

ax.set_xlabel(
    "Immune niche"
)

ax.set_ylabel(
    "Number of spots"
)

ax.set_title(
    "CellCharter niche abundance"
)

plt.tight_layout()

plt.savefig(
    output_path_figures
    / "10_niche_abundance.png",
    bbox_inches="tight",
    dpi=300
)

plt.close()

In [189]:
# -- Immune state composition by niche
fig, ax = plt.subplots(
    figsize=(9, 6)
)

sns.heatmap(
    immune_state_niche,
    cmap="viridis",
    annot=True,
    fmt=".2f",
    ax=ax
)

ax.set_xlabel(
    "Immune infiltration class"
)

ax.set_ylabel(
    "CellCharter niche"
)

ax.set_title(
    "Immune-class composition of spatial niches"
)

plt.tight_layout()

plt.savefig(
    output_path_figures
    / "10_niche_immune_class_composition.png",
    bbox_inches="tight",
    dpi=300
)

plt.close()

In [190]:
# -- Immune cell enrichment within niches
fig, ax = plt.subplots(
    figsize=(12, 7)
)

sns.heatmap(
    celltype_enrichment,
    cmap="coolwarm",
    center=1,
    ax=ax
)

ax.set_xlabel(
    "Immune cell type"
)

ax.set_ylabel(
    "CellCharter niche"
)

ax.set_title(
    "Immune cell enrichment within spatial niches"
)

plt.xticks(
    rotation=45,
    ha="right"
)

plt.tight_layout()

plt.savefig(
    output_path_figures
    / "10_niche_celltype_enrichment.png",
    bbox_inches="tight",
    dpi=300
)

plt.close()

In [191]:
# -- Niche distribution across lesions
fig, ax = plt.subplots(
    figsize=(7, 6)
)

sns.heatmap(
    niche_sample,
    cmap="viridis",
    annot=True,
    fmt=".2f",
    ax=ax
)

ax.set_xlabel(
    "Lesion"
)

ax.set_ylabel(
    "Immune niche"
)

ax.set_title(
    "Spatial niche distribution by lesion"
)

plt.tight_layout()

plt.savefig(
    output_path_figures
    / "10_niche_distribution_by_lesion.png",
    bbox_inches="tight",
    dpi=300
)

plt.close()

In [192]:
niche_composition.round(2)

,B,CD4 T,CD8 T,DC,Mono-C,Mono-NC,NK-CD16+,NK-CD16-,TRM,Treg,γδ T
immune_niche,,,,,,,,,,,
0,0.16,0.390000,0.520000,0.04,0.04,0.03,0.14,0.24,0.240000,0.26,0.08
1,2.47,7.220000,20.940001,2.39,1.07,1.93,3.10,6.51,5.180000,15.44,1.54
2,10.07,24.110001,25.030001,3.17,1.55,2.28,6.96,14.66,9.730000,14.43,5.74
3,10.01,7.130000,42.580002,6.43,2.97,4.38,15.17,31.07,16.290001,29.26,12.78
4,1.32,4.060000,6.060000,0.74,0.42,0.69,0.77,1.56,1.850000,3.98,0.35
5,4.03,13.750000,6.680000,0.69,0.31,0.40,1.08,2.39,2.840000,2.79,0.86
6,4.65,7.880000,15.860000,2.17,0.95,1.49,5.13,10.53,6.890000,10.26,4.21


### Manual niche interpretation

Going to review the abundace and renrichment tables before assigning biological tables. Because CellCharter cluster numbes can change after rerunning the model, this mapping should always be treated as a checkpoint.

In [193]:
niche_labels = {
    "0": "Immune-depleted",
    "1": "Effector immune",
    "2": "Lymphoid-rich",
    "3": "Immune hotspot",
    "4": "Immune-cold",
    "5": "Adaptive-enriched",
    "6": "Diffuse immune"
}


observed_niches = set(
    adata_immune.obs[
        "immune_niche"
    ]
    .astype(str)
    .unique()
)

mapped_niches = set(
    niche_labels
)

if observed_niches != mapped_niches:

    missing = sorted(
        observed_niches
        - mapped_niches
    )

    extra = sorted(
        mapped_niches
        - observed_niches
    )

    raise ValueError(
        "Manual niche map does not match the current CellCharter result. "
        f"Missing mappings: {missing}; extra mappings: {extra}"
    )

adata_immune.obs[
    "immune_niche_label"
] = (
    adata_immune.obs[
        "immune_niche"
    ]
    .astype(str)
    .map(
        niche_labels
    )
)

adata_immune.obs[
    "immune_niche_label"
] = adata_immune.obs[
    "immune_niche_label"
].astype("category")

print(
    adata_immune.obs[
        "immune_niche_label"
    ].value_counts()
)

immune_niche_label
Effector immune      759
Immune-cold          577
Adaptive-enriched    570
Immune-depleted      548
Diffuse immune       507
Lymphoid-rich        300
Immune hotspot        87
Name: count, dtype: int64


In [194]:
# -- Spatial mapping with labels


# -- Squidpy uses categorical labels in sorted order
squidpy_niche_order = sorted(
    adata_immune.obs[
        "immune_niche_label"
    ]
    .dropna()
    .astype(str)
    .unique()
)

adata_immune.uns[
    "immune_niche_label_colors"
] = [
    niche_palette[label]
    for label in squidpy_niche_order
]

sq.pl.spatial_scatter(
    adata_immune,
    color="immune_niche_label",
    library_key="sample_id",
    spatial_key="spatial",
    size=2,
    img=False,
    ncols=2,
    figsize=(6, 6),
    legend_loc=None,
    title=[
        "BEME_346 lesion",
        "BEME_355G lesion"
    ]
)

legend_elements = [
    Patch(
        facecolor=niche_palette[label],
        label=label
    )
    for label in squidpy_niche_order
]

plt.gcf().legend(
    handles=legend_elements,
    loc="center right",
    title="Immune niche",
    bbox_to_anchor=(1.12, 0.5)
)

plt.savefig(
    output_path_figures
    / "10_niche_discovery.png",
    bbox_inches="tight",
    dpi=300
)

plt.close()

In [195]:
# -- SAVE

output_file = (
    output_path_data
    / "10_immune_niches.h5ad"
)

adata_immune.write_h5ad(
    output_file
)

niche_key = pd.DataFrame(
    {
        "cluster": list(
            niche_labels.keys()
        ),
        "label": list(
            niche_labels.values()
        )
    }
)

niche_key.to_csv(
    output_path_results
    / "10_immune_niche_labels.csv",
    index=False
)

niche_composition.to_csv(
    output_path_results
    / "10_niche_mean_abundance.csv"
)

celltype_enrichment.to_csv(
    output_path_results
    / "10_niche_celltype_enrichment.csv"
)

immune_state_niche.to_csv(
    output_path_results
    / "10_niche_immune_class_composition.csv"
)

niche_sample.to_csv(
    output_path_results
    / "10_niche_distribution_by_lesion.csv"
)

saved = sc.read_h5ad(
    output_file,
    backed="r"
)

print(saved)

saved.file.close()

AnnData object with n_obs × n_vars = 3348 × 22220 backed at '/content/drive/MyDrive/endo-immune-atlas/data/interim/spatial/10_immune_niches.h5ad'
    obs: 'in_tissue', 'array_row', 'array_col', 'library_id', 'sample_id', 'gsm_id', 'tissue_type', 'condition', 'lesion_site', '_indices', '_scvi_batch', '_scvi_labels', 'meanscell_abundance_w_sf_B', 'meanscell_abundance_w_sf_CD4 T', 'meanscell_abundance_w_sf_CD8 T', 'meanscell_abundance_w_sf_DC', 'meanscell_abundance_w_sf_Mono-C', 'meanscell_abundance_w_sf_Mono-NC', 'meanscell_abundance_w_sf_NK-CD16+', 'meanscell_abundance_w_sf_NK-CD16-', 'meanscell_abundance_w_sf_TRM', 'meanscell_abundance_w_sf_Treg', 'meanscell_abundance_w_sf_γδ T', 'Total immune', 'B', 'CD4 T', 'CD8 T', 'DC', 'Mono-C', 'Mono-NC', 'NK-CD16+', 'NK-CD16-', 'TRM', 'Treg', 'γδ T', 'leiden_0.2', 'leiden_0.4', 'leiden_0.6', 'leiden_0.8', 'region_cluster', 'total_immune_spot_level', 'total_immune_region_level', 'tissue_level_immune_class', 'projected_senescence_score', 'cluster_

## Interpretation checkpoint

The niche labels should be based on the relative immune-cell enrichment profiles, not cluster number alone. Because only two lesions are included, lesion-specific abundance differences should be described as observations rather than generalized niche frequencies.

The joint CellCharter model allows the same niche identity to be compared across lesions, while the lesion-distribution table shows whether each niche is shared or sample-enriched.
